# MI 702 — Analyzing a Tn-seq screen

**Group A *Streptococcus* versus a human antibacterial enzyme**

A saturated *Streptococcus pyogenes* M1T1 5448 transposon library was exposed to
**human group IIA secreted phospholipase A₂ (hGIIA)** — a bactericidal enzyme
released by host cells during infection — at 0.125 µg/ml for one hour, then
allowed to recover. Four treated and four untreated cultures were sequenced.

Your job: find which genes change the bacterium's susceptibility to this enzyme,
then work out how far the data can actually be trusted.

The published study is van Hensbergen et al. 2018, *PLoS Pathog* 14(10):e1007348.
**Do not read its results section until you have finished Part C.**

---

### How to work in this notebook

- Run the cells in order. **Run the setup cell first** — nothing works without it.
- Cells marked **FALLBACK** load a finished file instead of computing it. Use one
  if a step fails or takes too long. Say in your write-up which you used; you lose
  no credit for it.
- Type your written answers into the `> **Your answer:**` cells.
- This notebook is your own copy. If you break something, close it without saving
  and re-open from the course link — the original is untouched.

**Generative AI is not permitted for any part of this assignment**, per the course
syllabus.

### Runtime

Part B does real read processing in Python. The times below were **measured on
this dataset**; Colab's free tier is often slower than the machine they were
measured on, so allow up to about twice these figures.

| Step | Measured | Allow up to |
|---|---|---|
| B1 demultiplex | 80 s | ~3 min |
| B2 trim (8 samples) | 15 s | ~30 s |
| B3 build index + map | 2.5 s | ~10 s |
| Everything else | under 1 s | a few s |

Downloads depend on your connection: the core data is 3.9 MB, the reads 156 MB.

## Setup

Run this once. It installs the course package and downloads the data.

In [ ]:
# Install the course analysis package and fetch the dataset.
!pip install -q "git+https://github.com/nkorotkova-uky/mi702-tnseq.git@v1.0"

import mi702tnseq as mi
from mi702tnseq import pipeline as P, stats as S, plots as PL, data as D
import numpy as np, pandas as pd, matplotlib.pyplot as plt

D.fetch("core")          # ~4 MB: genome, annotation, TA sites, tables, insertion counts
print("package version:", mi.__version__)
print("samples:", P.SAMPLES)

---
# Part A — What is in the data?

Before analyzing anything, establish what a read looks like and what the library
actually covers.

### A1. Read structure

`describe_reads` summarizes the first 50,000 reads of the multiplexed file. Run
it and look at the output before reading on.

In [ ]:
D.fetch("reads")        # ~156 MB, needed from here through Part B
arch = P.describe_reads("mi702_data/reads/ALL_SAMPLES_multiplexed.fastq.gz", n=50000)
for k, v in arch.items():
    print(f"{k}: {v}")

**A1 questions.** How long are the reads? What fraction carries a recognized
barcode, and what fraction contains the transposon sequence? Look at
`junction_dinucleotides` — one dinucleotide dominates overwhelmingly. Which, and
why that one?

> **Your answer:**
>
> _(replace this line)_

### A2. Anatomy of a single read

`dissect_read` splits one read into its parts. Run it on a few reads.

In [ ]:
for i, (head, seq, qual) in enumerate(P.read_fastq(
        "mi702_data/reads/ALL_SAMPLES_multiplexed.fastq.gz", limit=400)):
    parts = P.dissect_read(seq)
    if parts and len(parts["genomic_dna"]) == 15:
        print("read:", seq)
        for k, v in parts.items():
            print(f"   {k:26s} {v}")
        break

**A2 questions.** Only a short piece of each 75 bp read is informative. Which
piece, and how long is it? The quality scores decay towards the 3' end of Illumina
reads — explain why that matters much less here than it would in an RNA-seq
experiment.

Then the important one: the insertion site is the `TA` at the **3' end** of the
genomic tag, not at the read start. If you mistakenly recorded the insertion at
the start of the tag instead, by how many bases would every insertion be wrong,
and would the error be in a consistent direction?

> **Your answer:**
>
> _(replace this line)_

### A3. How saturated is this library?

The transposon inserts only at `TA` dinucleotides, so those are the only positions
that can ever carry an insertion.

In [ ]:
genome   = P.load_genome(D.path("GAS5448_CP008776.fasta"))
ta_sites = D.load_ta_sites()
print(f"genome length      : {len(genome):,} bp")
print(f"TA dinucleotides   : {len(ta_sites):,}")
print(f"mean spacing       : {len(genome)/len(ta_sites):.1f} bp")

wig = P.read_wig(D.path("wig", "control_rep1.wig"))
hit = sum(1 for v in wig.values() if v > 0)
print(f"\nTA sites with >=1 insertion in control_rep1: {hit:,} "
      f"({100*hit/len(ta_sites):.1f}% of all TA sites)")

**A3 questions.** How many TA sites exist, and what fraction carries an insertion
in this sample?

**A4. The trap.** The published library was described as *saturated*, yet your
answer to A3 is a few percent. Both statements are true. Explain why — and state
precisely what quantity your A3 percentage actually measures. (Hint: you were
given 300,000 reads per sample. The published experiment had about 31.6 million.)

> **Your answer:**
>
> _(replace this line)_

---
# Part B — From reads to insertion counts

Four steps: split the samples, trim away everything that is not genomic DNA, map
the tags to insertion sites, and count insertions per gene.

Each step prints a number. **Compare it with the checkpoint** before moving on —
a mismatch tells you something specific.

### B1. Demultiplex

All eight samples are in one file, separated by a 6 bp inline barcode.

> **Checkpoint:** 2,400,000 reads read, 300,000 assigned to each of the eight
> samples, and essentially nothing unrecognized. If many reads have an
> unrecognized barcode, the barcode length is wrong.

In [ ]:
tally, per_sample = P.demultiplex(
    "mi702_data/reads/ALL_SAMPLES_multiplexed.fastq.gz", "demux")
print(tally)
print(per_sample)

In [ ]:
# FALLBACK for B1 — the demultiplexed files are already in the dataset.
# import shutil; shutil.copytree("mi702_data/reads", "demux", dirs_exist_ok=True)

### B2. Trim

Remove the barcode from the 5' end and the transposon and adapter from the 3' end,
**keeping the TA**. What survives is the genomic tag that gets mapped.

> **Checkpoint:** 73–76% of reads survive, and the surviving tags are 15–17 nt.
> A retention near 100% means the transposon was never removed — you would then
> be mapping transposon sequence, which matches nothing in the genome.

In [ ]:
trim_stats = {}
for s in P.SAMPLES:
    total, kept = P.trim(f"demux/{s}.fastq.gz", f"trimmed/{s}.fastq.gz")
    trim_stats[s] = (total, kept, round(100 * kept / total, 1))
    print(f"{s:14s} {total:>7,} -> {kept:>7,}  ({trim_stats[s][2]}% kept)")

In [ ]:
# FALLBACK for B2
# import shutil; shutil.copytree("mi702_data/trimmed", "trimmed", dirs_exist_ok=True)
# (files there are named <sample>_trimmed.fastq.gz)

### B3. Map insertions to TA sites

Insertions can only land at TA sites, so instead of a general-purpose aligner we
index what a tag ending at each TA site should look like, on both strands, and
match exactly.

> **Checkpoint:** 94–95% of trimmed tags map to exactly one site. A rate far
> below this means the barcode boundary is off by one base — suspect that before
> suspecting the reference genome.

In [ ]:
index = P.build_ta_index(genome, ta_sites, k=12)
print(f"index entries: {len(index):,}")

insertions, mapping_stats = {}, {}
for s in P.SAMPLES:
    insertions[s], mapping_stats[s] = P.map_insertions(
        f"trimmed/{s}.fastq.gz", genome, index, k=12)
    st = mapping_stats[s]
    pct = 100 * st["unique"] / sum(st.values())
    print(f"{s:14s} unique {st['unique']:>7,} ({pct:.1f}%)   "
          f"TA sites hit {len(insertions[s]):>6,}")

In [ ]:
# FALLBACK for B3 — load the provided insertion counts instead.
# insertions = {s: P.read_wig(D.path("wig", f"{s}.wig")) for s in P.SAMPLES}

### B4. Count insertions per gene

Insertions are summed over the first 90% of each gene. The published analysis
discarded the final 10% because an insertion very near the 3' end often leaves a
working protein.

> **Checkpoint:** a (8, 1080) matrix with a median of about 23 reads per gene
> per sample.

In [ ]:
genes = D.load_published_table()
counts = np.vstack([P.count_per_gene(insertions[s], genes) for s in P.SAMPLES])
print("count matrix:", counts.shape)
print("median reads per gene per sample:", int(np.median(counts)))
print("genes with at least one read in all eight samples:",
      int((counts > 0).all(axis=0).sum()), "of", counts.shape[1])

### B5. Look at the data

Before computing statistics, look at a region. This is the notebook equivalent of
a genome browser: one lane per sample, a vertical line at each insertion.

The *dlt* operon modifies the cell envelope's surface charge. Run the cell, then
try `lytR/cpsA` or a gene you expect to be unaffected.

In [ ]:
row = genes[genes["gene_name"] == "dltA"].iloc[0]
fig, axes = PL.insertion_track(insertions,
                               int(row["start_90pct"]) - 3000,
                               int(row["stop_90pct"]) + 3000,
                               genes=genes)
plt.show()

**B5 questions.** Describe what you see across the *dlt* genes: do the treated
lanes differ from the control lanes, and in which direction? Now pick a gene you
expect to be unaffected and plot it too. What does the comparison between the two
regions tell you that a table of numbers would not?

Save both figures for your write-up.

*(The provided `.wig` files also load into the IGV web app at igv.org/app if you
would rather use a genome browser — upload the genome FASTA, the GFF3 and the wig
files. That is optional.)*

> **Your answer:**
>
> _(replace this line)_

---
# Part C — Which genes matter?

You will analyze the same biological question three ways and get three different
answers. Understanding **why** they differ is the point of this part.

### C1. A permutation test on your own data

A resampling test asks: if the treatment labels were meaningless, how often would
we see a difference this large? It makes no assumption about the distribution of
the data, which sounds strictly better.

Run it, then read the question below before concluding anything.

In [ ]:
norm = counts / (counts.sum(axis=1, keepdims=True) / counts.sum(axis=1).mean())
result = S.resampling_test(norm[:4], norm[4:], exact=True)
print("distinct splits used :", result["permutations_used"])
print("smallest p observed  :", round(result["p"].min(), 4))
print("genes at q < 0.05    :", int((result["q"] < 0.05).sum()))
print("smallest q observed  :", round(result["q"].min(), 3))

**C1. The surprising result.** You almost certainly got **zero** significant
genes. Before blaming the data, work out the arithmetic.

(a) You have 4 control and 4 treated samples. In how many distinct ways can 8
samples be split into two groups of 4? (Swapping the two group labels gives the
same split, so count each split once.)

(b) A permutation p-value is the fraction of splits giving a difference at least
as large as the real one. The real split is always one of them. What is therefore
the **smallest p-value this test can possibly produce**?

(c) Many genes will be tied at exactly that smallest p-value. Count how many
(`int((result["p"] <= result["p"].min() + 1e-12).sum())`). Benjamini–Hochberg
divides by rank, so k genes sharing the minimum p all receive
q = p_min x n_tests / k. Work out the q those tied genes get, and check it against
the smallest q you actually observed.

(d) Given that, how many genes would have to tie at the floor before *anything*
could clear q < 0.05? Is that plausible? The cell below checks your arithmetic.

(e) Would sequencing ten times more reads fix this? Answer yes or no and justify
it in one sentence. Then say what *would* fix it.

In [ ]:
n_floor = int((result["p"] <= result["p"].min() + 1e-12).sum())
print("genes tied at the floor:", n_floor)
print(S.permutation_floor(n_a=4, n_b=4, n_tests=1080, n_at_floor=n_floor))
print("\nsame experiment with 6 replicates per group:")
print(S.permutation_floor(n_a=6, n_b=6, n_tests=1080, n_at_floor=n_floor))

> **Your answer:**
>
> _(replace this line)_

### C2. A parametric test on your own data

A t-test assumes a distribution but is not limited by the number of label
permutations. Same data, different method.

In [ ]:
tt = S.welch_or_student_test(counts[:4], counts[4:], log_transform=True)
sig = tt["q"] < 0.05
print("genes at q < 0.05:", int(sig.sum()))
out = genes.loc[sig, ["locus_tag", "gene_name", "putative_function"]].copy()
out["q"] = np.round(tt["q"][sig], 4)
print(out.to_string(index=False))

> **Your answer:**
>
> _(replace this line)_

### C3. The published full-depth data

`GAS_hGIIA_gene_RPKM.csv` is the per-gene table deposited with the paper: the same
eight samples at roughly 31.6 million reads each, already normalized to RPKM. It
contains **no** fold changes and **no** p-values — you compute those.

In [ ]:
C = genes[[f"control_rep{i}" for i in (1, 2, 3, 4)]].to_numpy(float)
H = genes[[f"hGIIA_rep{i}"   for i in (1, 2, 3, 4)]].to_numpy(float)

fold = S.signed_fold_change(H.mean(axis=1), C.mean(axis=1))
pub_tt = S.welch_or_student_test(C.T, H.T, log_transform=True)
print("genes at q < 0.05:", int((pub_tt["q"] < 0.05).sum()))

log2fc = np.where(fold > 0, np.log2(np.abs(fold)), -np.log2(np.abs(fold)))
fig, ax = PL.volcano(log2fc, pub_tt["q"],
                     labels=genes["gene_name"].to_numpy(dtype=object),
                     label_genes=["dltA", "dltB", "gacI", "gacH", "lytR/cpsA"],
                     title="hGIIA vs control, published full-depth data")
plt.show()

**C3 questions.** How many genes are significant, and which are they? Report them
with fold changes and directions, and state clearly which direction means
*more susceptible* to hGIIA. Save the volcano plot.

Note the fold-change convention: it is a ratio, but values below 1 are written as
the negative reciprocal, so −4 means fourfold depleted.

> **Your answer:**
>
> _(replace this line)_

### C4. Three answers, one dataset

Collect what you got:

| Analysis | Data | Genes at q < 0.05 |
|---|---|---|
| C1 permutation | your subset | ? |
| C2 t-test | your subset | ? |
| C3 t-test | published full depth | ? |

The paper itself reports **17** genes using a *regularized* t-test (Cyber-T),
which borrows variance information across genes instead of estimating each gene's
variance from its own four values.

**C4 questions.** Explain the difference between C1 and C2 — same data, same
question, different answer. Then explain the difference between C2 and C3. Finally:
your C3 list came from the paper's own data but probably does not match the
paper's 17 genes. Does that mean one of you is wrong? What does a difference
between two statistical methods on identical data actually tell you?

> **Your answer:**
>
> _(replace this line)_

---
# Part D — Interpretation

Written answers. No new computation required, though you may compute if it helps.

**D1. The biology.** Among the depleted genes are the *dlt* operon and `lytR/cpsA`;
among the enriched are `gacH` and `gacI`. Look up what these do. hGIIA is a small,
strongly cationic enzyme that must reach and hydrolyze membrane phospholipids.

Propose a single mechanism that explains why losing one set of genes makes the
bacterium more vulnerable while losing the other makes it more resistant. Your
explanation should account for both directions, not just one.

> **Your answer:**
>
> _(replace this line)_

**D2. The genes that are not there.** Run the cell. The annotation contains 1,760
coding sequences; the results table contains 1,080 genes.

In [ ]:
annotation = D.load_annotation()
in_table = set(genes["locus_tag"])
missing = annotation[~annotation["locus_tag"].isin(in_table)]
print("annotated CDS      :", len(annotation))
print("genes in the table :", len(genes))
print("annotated but absent from the table:", len(missing))
print("\nnote this is NOT simply 1760 - 1080 — explain the discrepancy")
print("\nmedian CDS length, absent  : %.0f bp" % (missing["end"] - missing["start"]).median())
present = annotation[annotation["locus_tag"].isin(in_table)]
print("median CDS length, present : %.0f bp" % (present["end"] - present["start"]).median())
print("\nmost common products among the absent genes:")
print(missing["product"].value_counts().head(8).to_string())

**D2 questions.**

(a) How many annotated genes have no row in the results table? Explain why this is
not simply 1,760 − 1,080.

(b) Give **two distinct reasons** a gene would be absent, and say which reason the
length comparison and the product list support. The two reasons have very
different consequences — explain what each means for interpreting the screen.

(c) What fraction of the genome did this screen effectively decline to assess?
Does that change how you would describe its conclusions?

> **Your answer:**
>
> _(replace this line)_

**D3. An alternative explanation.** The *dlt* genes form an operon. A transposon
inserted in `dltA` may stop transcription of everything downstream.

Why does this complicate the claim that *dltA* specifically is required for hGIIA
resistance? Describe one experiment that would distinguish a genuine *dltA*
requirement from a polar effect on the rest of the operon.

> **Your answer:**
>
> _(replace this line)_

**D4. Count them yourself.** The paper's abstract states a number of
hGIIA-susceptible genes. Using the deposited table, count how many genes pass
q < 0.05 in the susceptible direction according to the paper's own corrected
p-values.

In [ ]:
key_cols = ["locus_tag", "gene_name", "putative_function"]
# The deposited table has no p-values; use YOUR C3 result for the direction count.
susceptible = (pub_tt["q"] < 0.05) & (log2fc < 0)
resistant   = (pub_tt["q"] < 0.05) & (log2fc > 0)
print("your susceptible count:", int(susceptible.sum()))
print("your resistant count  :", int(resistant.sum()))
print(genes.loc[susceptible, key_cols].to_string(index=False))

**D4 questions.** Compare your count with the number stated in the paper's
abstract. If they differ, suggest why, and say which number you would cite and
how you would phrase it.

> **Your answer:**
>
> _(replace this line)_

**D5. One design change.** You are repeating this screen in a mouse model of skin
infection rather than in a tube.

Name the single biggest threat to the experiment, explain the mechanism by which
it would corrupt your results, and give one concrete thing you would do about it.

> **Your answer:**
>
> _(replace this line)_

---
## Submission

Export this notebook as PDF (**File → Print → Save as PDF**) with all cells run
and outputs visible, plus your written answers filled in. Include:

- the insertion-track figures from B5
- the volcano plot from C3
- your checkpoint numbers from Part B alongside the expected values
- a statement of any **FALLBACK** cells you used and why

### How this is graded

Five of the ten MI 702 competency areas, on the Beginner → Advanced scale:

| Competency area | Where |
|---|---|
| 2. Understanding of the relevant method or platform | A1, A2, A4, B3 |
| 4. Data processing, workflow, or analytical execution | B1–B4 |
| 6. Quantitative reasoning and statistical judgment | C1–C4 |
| 5. Data interpretation and evidence-based conclusions | B5, D1, D4 |
| 8. Critical evaluation of artifacts, limitations, alternatives | A4, C4, D2, D3, D5 |

Getting the "right" gene list matters less than showing you know what your numbers
can and cannot support. A student who reaches a defensible conclusion with a
clearly stated method and an honest account of its limits scores higher than one
who reproduces the paper's numbers without understanding them.